In [1]:
%cd ..

c:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions


In [2]:
from dotenv import load_dotenv

load_dotenv(".env.ambari")


True

In [3]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [4]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [5]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [6]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "/opt/datasets/crawlers/vcs/survicate/data"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "cx_survicate_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/cx_survicate_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/cx_survicate_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/cx_survicate_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    return False


In [7]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_csv_and_normalize_columns(
    file_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_csv(file_path, header=None, encoding="utf-8", dtype=str)
    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(str(col).strip())

            # 2️⃣ fallback snake_case
            if base is None:
                base = snake_case(col)
                print(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)
    df = df.fillna("")

    return df

def read_folder_and_union_csv(
    folder_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0,
    encoding="utf-8"
):
    dfs = []
    folder = Path(folder_path)

    files = sorted(folder.glob("*.csv"))
    if not files:
        raise ValueError("❌ Không tìm thấy file CSV nào trong folder")

    for file in files:
        df = read_csv_and_normalize_columns(
            file_path=file,
            mapping=mapping,
            drop_rows=drop_rows,
            header_row=header_row
        )

        # trace file nguồn
        df["source_file"] = file.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [8]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [9]:

SURVEY_DICT = {
    "survey_id": "survey_id",
    "survey_name": "survey_name",
    "created_at": "created_at",
    "Folder": "folder",
    "question_count": "question_count",
    "response_count": "response_count",
    "Survey_active": "survey_active",
    "Journey_id": "journey_id",
    "Journey": "journey",
    "Điểm chạm": "touchpoint",
    "SPDV": "product_service",
    "Note": "note",
    "Quyền tạo ticket cho negative feedback":"allow_negative_feedback_ticket"
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\cx-cso\survicate_export v2.xlsx"
resource_name = "excel_surveys"

df = pd.read_excel(filename, sheet_name="Surveys", dtype=str)

# Remove whitespace ở header nếu có
df.columns = df.columns.str.strip()
# Rename
df = df[[col for col in df.columns if col in SURVEY_DICT.keys()]]
df = df.rename(columns=SURVEY_DICT)

df["source_file"] =  filename.split("\\")[-1]
df = df.fillna('')
print(df.columns)
print(df.dtypes)
df.head(2)

Index(['survey_id', 'survey_name', 'created_at', 'folder', 'question_count',
       'response_count', 'survey_active', 'journey_id', 'journey',
       'touchpoint', 'product_service', 'note',
       'allow_negative_feedback_ticket', 'source_file'],
      dtype='object')
survey_id                         object
survey_name                       object
created_at                        object
folder                            object
question_count                    object
response_count                    object
survey_active                     object
journey_id                        object
journey                           object
touchpoint                        object
product_service                   object
note                              object
allow_negative_feedback_ticket    object
source_file                       object
dtype: object


,survey_id,survey_name,created_at,folder,question_count,response_count,survey_active,journey_id,journey,touchpoint,product_service,note,allow_negative_feedback_ticket,source_file
0,d4d685c36e4e1444,Khảo sát thử nghiệm app TNXH_102025,2025-10-09T10:54:26.000000Z,,8,3,No,,,,,,No,survicate_export v2.xlsx
1,a5b38bdd86a1d42d,Testing_survey,2025-10-08T09:50:25.000000Z,,4,0,No,,,,,,No,survicate_export v2.xlsx


In [10]:
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

✅ Done: excel_surveys.json created
Start crawl :  excel_surveys
excel_surveys
Replace Upload  /opt/datasets/crawlers/vcs/survicate/data/excel_surveys ./tmp/data/cx_survicate_raw/excel_surveys/data_excel_surveys_20260422_110117.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/survicate/data/excel_surveys
Uploaded SQL definition


False

In [11]:

SURVEY_QUESTION_DICT = {
    "survey_id": "survey_id",
    "survey_name": "survey_name",
    "question_id": "question_id",
    "question_type": "question_type",
    "question_text": "question_text",
    "answer_choice_id": "answer_choice_id",
    "answer_choice_content": "answer_choice_content",
    "question_type": "question_type",
    "active": "active",
    "KPI": "kpi",
    "SPDV": "product_service",
    "Layer": "layer",
    "KPI Checked": "kpi_checked"
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\cx-cso\survicate_export v2.xlsx"
resource_name = "excel_survey_questions"

df = pd.read_excel(filename, sheet_name="Survey_Questions (2)", dtype=str)

# Remove whitespace ở header nếu có
df.columns = df.columns.str.strip()
# Rename
df = df[[col for col in df.columns if col in SURVEY_QUESTION_DICT.keys()]]
df = df.rename(columns=SURVEY_QUESTION_DICT)

df["source_file"] = filename.split("\\")[-1]
df = df.fillna('')
print(df.columns)
print(df.dtypes)
df.head(2)

Index(['survey_id', 'survey_name', 'question_id', 'question_type',
       'question_text', 'answer_choice_id', 'answer_choice_content', 'active',
       'kpi', 'product_service', 'layer', 'kpi_checked', 'source_file'],
      dtype='object')
survey_id                object
survey_name              object
question_id              object
question_type            object
question_text            object
answer_choice_id         object
answer_choice_content    object
active                   object
kpi                      object
product_service          object
layer                    object
kpi_checked              object
source_file              object
dtype: object


,survey_id,survey_name,question_id,question_type,question_text,answer_choice_id,answer_choice_content,active,kpi,product_service,layer,kpi_checked,source_file
0,429754b83f3c3f93,[2025] PENTEST + AUDIT,3211426,rating,Quý khách vui lòng đánh giá mức độ hài lòng vớ...,8310164,1,No,CSAT SPDV,Pentest,,,survicate_export v2.xlsx
1,429754b83f3c3f93,[2025] PENTEST + AUDIT,3211427,multiple,Quý khách ĐÁNH GIÁ CAO yếu tố nào của Dịch vụ ...,8310169,Nội dung khảo sát trước,No,Tính năng good,Pentest,,,survicate_export v2.xlsx


In [12]:
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

✅ Done: excel_survey_questions.json created
Start crawl :  excel_survey_questions
excel_survey_questions
Replace Upload  /opt/datasets/crawlers/vcs/survicate/data/excel_survey_questions ./tmp/data/cx_survicate_raw/excel_survey_questions/data_excel_survey_questions_20260422_110118.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/survicate/data/excel_survey_questions
Uploaded SQL definition


False